# Beam-orientation comparisons

Compare p-going, Pb-going, and combined-orientation eta distributions at configurable Gen, Ref, and Reco levels. Each two-dimensional single-jet or dijet histogram is projected onto eta in the configured $p_{T}$ or $p_{T}^{ave}$ intervals. Direction-comparison figures contain an overlay and a ratio panel; the dedicated Gen/Reco diagnostic uses side-by-side panels, and same-direction frame overlays intentionally do not form ratios. Lab-unflipped direction comparisons use p-going as the ratio denominator; flipped-lab and CM comparisons additionally include the combined file and use it as the denominator.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    COMMON_ETA_CM_RANGE, DIJET_DELTA_PHI_SELECTION_LABEL,
    DIJET_PTAVE_BINS, SINGLE_JET_PT_BINS,
)
from hist_analysis.python.histogram_io import resolve_combined_file, resolve_direction_file
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style, set_legend_style, set_pad_style,
    style_single_panel_axes,
)

## Configuration

`NORMALIZATION='integral'` compares eta shapes rather than the stored direction-dependent yields. Set it to `'none'` to compare yields or `'bin_width'` for density normalization. `REBIN_X` and `REBIN_Y` are common factors applied to the pT/pTave and eta axes of every source TH2 before projection. The default factors preserve the original bins; note that ROOT moves a remainder into overflow when a factor does not divide the axis bin count. Projection intervals are half-open, `[low, high)`. Each frame configures its displayed `x_range`; for direction comparisons, the shared renderer applies the CM range consistently to the overlay, ratios, and horizontal reference line before saving. The exact unsuffixed histogram keys requested for this study are used; no fallback keys are substituted.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
FILE_STEM = 'jetId'
LEVELS = ('Gen', 'Reco')
# LEVELS = ('Gen', 'Ref', 'Reco')
FRAME_OVERLAY_DIRECTION = 'pgoing'  # pgoing or Pbgoing
FRAME_OVERLAY_LEVEL = 'Gen'          # Gen or Reco
NORMALIZATION = 'integral'    # none, integral, or bin_width
REBIN_X = 1                   # common pT or pTave-axis factor
REBIN_Y = 2                   # common eta-axis factor
RATIO_RANGE = (0.8, 1.2)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'beam_orientation'

DIRECTION_LABELS = {
    'pgoing': 'p-going',
    'Pbgoing': 'Pb-going',
    'combined': 'combined',
}
FRAME_CONFIG = {
    # 'lab_unflipped_unvweighted': {
    #     'suffix': 'PtEtaLabUnflippedUnvweighted',
    #     'directions': ('pgoing', 'Pbgoing'),
    #     'nominal': 'p-going',
    #     'frame_label': 'Lab frame (unfl., unvweighted)',
    #     'x_range': None,
    # },
    'lab_unflipped': {
        'suffix': 'PtEtaLabUnflipped',
        'directions': ('Pbgoing', 'pgoing' ),
        'nominal': 'p-going',
        'frame_label': 'Lab frame (unfl.)',
        'x_range': None,
    },
    'lab_flipped': {
        'suffix': 'PtEtaLab',
        'directions': ('Pbgoing', 'pgoing', 'combined'),
        'nominal': 'combined',
        'frame_label': 'Lab frame (flipped)',
        'x_range': None,
    },
    'cm': {
        'suffix': 'PtEtaCM',
        'directions': ('Pbgoing', 'pgoing', 'combined'),
        'nominal': 'combined',
        'frame_label': 'CM frame',
        'x_range': (-2.4, 2.4),
    },
}

def mc_file(direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
    return resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM)

input_files = {direction: mc_file(direction)
               for direction in ('Pbgoing', 'pgoing', 'combined')}
missing = [str(path) for path in input_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing configured ROOT files:\n' + '\n'.join(missing))
input_files

## Comparison helper

The histogram key is assembled only from the requested level, jet category, and frame suffix. The notebook's `root_eta_projections` helper validates the physical pT/pTave and eta axes from their ranges, projects eta, checks shared bin edges, and applies the selected normalization. The shared `draw_closure` renderer then draws the overlay, nontrivial ratios, annotations, configured x-range, and reference line.

In [ ]:
def root_eta_projections(histogram_specs, pt_range, name_prefix):
    if NORMALIZATION not in ('none', 'integral', 'bin_width'):
        raise ValueError(f'Unsupported normalization: {NORMALIZATION!r}')
    pt_low, pt_high = pt_range
    histograms = {}
    keys = {}
    for index, (label, filename, histogram_key) in enumerate(histogram_specs):
        root_file = ROOT.TFile.Open(str(filename), 'READ')
        if not root_file or root_file.IsZombie():
            raise OSError(f'Could not open {filename}')
        source_in_file = root_file.Get(histogram_key)
        if not source_in_file or not source_in_file.InheritsFrom('TH2'):
            root_file.Close()
            raise KeyError(f'{histogram_key} is missing or is not a TH2 in {filename}')
        source = source_in_file.Clone(f'{name_prefix}_source_{index}')
        source.SetDirectory(0)
        root_file.Close()

        x_axis = source.GetXaxis()
        y_axis = source.GetYaxis()
        x_is_pt = x_axis.GetXmax() > 20.0
        y_is_eta = y_axis.GetXmin() >= -10.0 and y_axis.GetXmax() <= 10.0
        if not (x_is_pt and y_is_eta):
            raise ValueError(
                f'Expected ROOT TH2 axes (pT, eta) for {histogram_key}; got '
                f'X=({x_axis.GetXmin()}, {x_axis.GetXmax()}), '
                f'Y=({y_axis.GetXmin()}, {y_axis.GetXmax()})'
            )
        source.Rebin2D(REBIN_X, REBIN_Y)
        pt_axis = source.GetXaxis()
        first_pt_bin = max(1, pt_axis.FindBin(pt_low + 0.001))
        last_pt_bin = min(pt_axis.GetNbins(), pt_axis.FindBin(pt_high - 0.001))
        if first_pt_bin > last_pt_bin:
            raise ValueError(f'Empty pT interval [{pt_low}, {pt_high}) for {histogram_key}')
        projection = source.ProjectionY(
            f'{name_prefix}_eta_{index}', first_pt_bin, last_pt_bin, 'e'
        )
        projection.SetDirectory(0)
        if NORMALIZATION != 'none':
            integral = projection.Integral(1, projection.GetNbinsX())
            if integral <= 0.0:
                raise ValueError(f'Cannot normalize empty projection for {label}')
            projection.Scale(
                1.0 / integral, 'width' if NORMALIZATION == 'bin_width' else ''
            )
        histograms[label] = projection
        keys[label] = histogram_key

    reference = next(iter(histograms.values()))
    reference_axis = reference.GetXaxis()
    for label, histogram in histograms.items():
        axis = histogram.GetXaxis()
        same_edges = histogram.GetNbinsX() == reference.GetNbinsX() and all(
            abs(axis.GetBinLowEdge(bin_index) - reference_axis.GetBinLowEdge(bin_index)) < 1e-9
            for bin_index in range(1, histogram.GetNbinsX() + 2)
        )
        if not same_edges:
            raise ValueError(f'Incompatible eta binning for {label}; ROOT projections were not altered')
    return histograms, keys

def orientation_specs(level, jet_kind, frame):
    if level not in LEVELS:
        raise ValueError(f'Unsupported level: {level!r}')
    if jet_kind not in ('single', 'dijet'):
        raise ValueError(f'Unsupported jet kind: {jet_kind!r}')
    config = FRAME_CONFIG[frame]
    object_name = 'InclusiveJet' if jet_kind == 'single' else 'Dijet'
    histogram_key = f'h{level}{object_name}{config["suffix"]}'
    return [
        (DIRECTION_LABELS[direction], input_files[direction], histogram_key)
        for direction in config['directions']
    ]

def apply_x_range(histograms, x_range):
    if x_range is None:
        return
    x_range_low, x_range_high = x_range
    for histogram in histograms:
        histogram.GetXaxis().SetRangeUser(x_range_low, x_range_high)

def object_label(level, jet_kind):
    object_type = 'jets' if jet_kind == 'single' else 'dijets'
    algorithm = ' (AK4PF)' if jet_kind == 'single' and level in ('Gen', 'Reco') else ''
    return f'{level} {object_type}{algorithm}'

def plot_headroom(jet_kind):
    return 1.6 if jet_kind == 'dijet' else 1.5

def eta_axis_title(jet_kind, frame=None):
    object_type = 'jet' if jet_kind == 'single' else 'dijet'
    cm_suffix = '_{CM}' if frame == 'cm' else ''
    return f'#eta^{{{object_type}}}{cm_suffix}'

def eta_y_axis_title(jet_kind, frame=None):
    if NORMALIZATION == 'integral':
        return f'dN/d{eta_axis_title(jet_kind, frame)}'
    if NORMALIZATION == 'bin_width':
        return f'1/N dN/d{eta_axis_title(jet_kind, frame)}'
    return 'Jets'

def plot_label_lines(level, jet_kind, pt_range, frame=None):
    low, high = pt_range
    momentum = 'p_{T}' if jet_kind == 'single' else 'p_{T}^{ave}'
    lines = [GENERATOR.capitalize(), object_label(level, jet_kind)]
    if frame is not None:
        lines.append(FRAME_CONFIG[frame]['frame_label'])
    lines.append(f'{low:g} < {momentum} < {high:g} GeV')
    if jet_kind == 'dijet':
        eta_cut = max(abs(edge) for edge in COMMON_ETA_CM_RANGE)
        lines.extend((
            f'|#eta^{{jet}}_{{CM}}| < {eta_cut:g}',
            'p_{T}^{Lead} > 50 GeV',
            'p_{T}^{SubLead} > 40 GeV',
            DIJET_DELTA_PHI_SELECTION_LABEL,
        ))
    return lines

def draw_orientation_comparison(level, jet_kind, frame, pt_range):
    config = FRAME_CONFIG[frame]
    histograms, keys = root_eta_projections(
        orientation_specs(level, jet_kind, frame), pt_range,
        f'orientation_{level}_{jet_kind}_{frame}',
    )
    low, high = pt_range
    tag = f'{GENERATOR}_{level.lower()}_{jet_kind}_{frame}_pt_{low:g}_{high:g}'
    canvas, ratios = draw_closure(
        histograms,
        config['nominal'],
        title='',
        x_title=eta_axis_title(jet_kind, frame),
        y_title=eta_y_axis_title(jet_kind, frame),
        ratio_range=RATIO_RANGE,
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{tag}.pdf',
        save_png=SAVE_PNG,
        draw_nominal_ratio=False,
        reference_line_x_range=config['x_range'],
        headroom=plot_headroom(jet_kind),
        canvas_name=tag,
        annotations=plot_label_lines(level, jet_kind, pt_range, frame),
        x_range=config['x_range'],
    )
    return {
        'canvas': canvas,
        'histograms': histograms,
        'ratios': ratios,
        'keys': keys,
        'nominal': config['nominal'],
    }

In [ ]:
# Inclusive Gen-jet eta projections for one frame, using ROOT throughout.
INCLUSIVE_GEN_FRAME = 'cm'
INCLUSIVE_GEN_PT_RANGE = SINGLE_JET_PT_BINS[2]

if INCLUSIVE_GEN_FRAME not in FRAME_CONFIG:
    raise ValueError(f'Unknown frame: {INCLUSIVE_GEN_FRAME!r}')

frame_config = FRAME_CONFIG[INCLUSIVE_GEN_FRAME]
histogram_key = f'hGenInclusiveJet{frame_config["suffix"]}'
pt_low, pt_high = INCLUSIVE_GEN_PT_RANGE
inclusive_gen_eta = {}

for index, direction in enumerate(frame_config['directions']):
    root_file = ROOT.TFile.Open(str(input_files[direction]), 'READ')
    if not root_file or root_file.IsZombie():
        raise OSError(f'Could not open {input_files[direction]}')
    source_in_file = root_file.Get(histogram_key)
    if not source_in_file or not source_in_file.InheritsFrom('TH2'):
        root_file.Close()
        raise KeyError(f'{histogram_key} is missing or is not a TH2 in {input_files[direction]}')
    source = source_in_file.Clone(f'inclusive_gen_source_{index}')
    source.SetDirectory(0)
    root_file.Close()

    source.Rebin2D(REBIN_X, REBIN_Y)
    pt_axis = source.GetXaxis()
    first_pt_bin = pt_axis.FindBin(pt_low + 0.001)
    last_pt_bin = pt_axis.FindBin(pt_high - 0.001)  # half-open [low, high)
    projection = source.ProjectionY(
        f'inclusive_gen_eta_{index}', first_pt_bin, last_pt_bin, 'e'
    )
    projection.SetDirectory(0)
    if NORMALIZATION != 'none':
        integral = projection.Integral()
        if integral <= 0.0:
            raise ValueError(f'Cannot normalize empty projection for {direction}')
        projection.Scale(1.0 / integral, 'width' if NORMALIZATION == 'bin_width' else '')
    inclusive_gen_eta[DIRECTION_LABELS[direction]] = projection

inclusive_reco_eta, inclusive_reco_keys = root_eta_projections(
    orientation_specs('Reco', 'single', INCLUSIVE_GEN_FRAME),
    INCLUSIVE_GEN_PT_RANGE,
    'inclusive_reco',
)

inclusive_gen_reco_canvas = ROOT.TCanvas(
    'inclusive_gen_reco_eta_canvas', '',
    DEFAULT_PLOT_STYLE.side_by_side_canvas_width,
    DEFAULT_PLOT_STYLE.side_by_side_canvas_height,
)
inclusive_gen_reco_canvas.Divide(2, 1)
inclusive_gen_reco_objects = []
panel_frame_histograms = []
for pad_index, (level, histograms) in enumerate(
    (('Gen', inclusive_gen_eta), ('Reco', inclusive_reco_eta)), start=1
):
    pad = inclusive_gen_reco_canvas.cd(pad_index)
    set_pad_style(pad, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    pad.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
    pad.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)
    legend = ROOT.TLegend(0.64, 0.70, 0.88, 0.88)
    set_legend_style(legend)
    panel_maximum = max(
        histogram.GetBinContent(bin_index) + histogram.GetBinError(bin_index)
        for histogram in histograms.values()
        for bin_index in range(1, histogram.GetNbinsX() + 1)
    )
    direction_style_indices = {'Pb-going': 1, 'p-going': 0, 'combined': 2}
    for index, (label, histogram) in enumerate(histograms.items()):
        set_1d_style(histogram, direction_style_indices[label])
        histogram.SetTitle('')
        histogram.GetXaxis().SetTitle(eta_axis_title('single', INCLUSIVE_GEN_FRAME))
        histogram.GetYaxis().SetTitle(eta_y_axis_title('single', INCLUSIVE_GEN_FRAME))
        style_single_panel_axes(histogram)
        histogram.SetMaximum(plot_headroom('single') * panel_maximum)
        histogram.Draw('E1' if index == 0 else 'E1 SAME')
        legend.AddEntry(histogram, label, 'p')
    legend.Draw()
    apply_x_range(histograms.values(), frame_config['x_range'])
    pad.Modified()
    labels = draw_text_block(
        pad, plot_label_lines(level, 'single', INCLUSIVE_GEN_PT_RANGE, INCLUSIVE_GEN_FRAME),
    )
    panel_frame_histograms.append(next(iter(histograms.values())))
    inclusive_gen_reco_objects.extend([pad, legend, *histograms.values(), *labels])
inclusive_gen_reco_canvas.Update()
shared_rendered_maximum = max(
    inclusive_gen_reco_canvas.GetPad(pad_index).GetUymax() for pad_index in (1, 2)
)
for pad_index, frame_histogram in enumerate(panel_frame_histograms, start=1):
    frame_histogram.SetMaximum(shared_rendered_maximum)
    inclusive_gen_reco_canvas.GetPad(pad_index).Modified()
inclusive_gen_reco_canvas.Update()
inclusive_gen_reco_canvas._objects = inclusive_gen_reco_objects
display(inclusive_gen_reco_canvas)

## Single-jet eta distributions in pT intervals

In [ ]:
single_jet_results = {}
for level in LEVELS:
    for frame in FRAME_CONFIG:
        for pt_range in SINGLE_JET_PT_BINS:
            result_key = (level, frame, pt_range)
            result = draw_orientation_comparison(level, 'single', frame, pt_range)
            single_jet_results[result_key] = result
            print(result_key, result['keys'], f"ratio denominator: {result['nominal']}")
            display(result['canvas'])

## Dijet eta distributions in pTave intervals

In [ ]:
dijet_results = {}
for level in LEVELS:
    for frame in FRAME_CONFIG:
        for ptave_range in DIJET_PTAVE_BINS:
            result_key = (level, frame, ptave_range)
            result = draw_orientation_comparison(level, 'dijet', frame, ptave_range)
            dijet_results[result_key] = result
            print(result_key, result['keys'], f"ratio denominator: {result['nominal']}")
            display(result['canvas'])

In [ ]:
# Fit and overlay the CM-frame beam-orientation ratios for every dijet pTave bin.
DIJET_CM_RATIO_FIT_FUNCTION = 'pol0'
DIJET_CM_RATIO_FIT_RANGE = COMMON_ETA_CM_RANGE  # selected |eta_CM^jet| range (e.g. 1.4, 1.5, 1.6)
DIJET_CM_RATIO_FIT_OPTIONS = 'RQ0S'
DIJET_CM_RATIO_SELECTION_TEXT_SIZE = DEFAULT_PLOT_STYLE.annotation_text_size
DIJET_CM_RATIO_SELECTION_TEXT_SIZE = 0.028

dijet_cm_ratio_fit_results = {}
for level in LEVELS:
    for ptave_range in DIJET_PTAVE_BINS:
        source_result = dijet_results[(level, 'cm', ptave_range)]
        low, high = ptave_range
        tag = f'{GENERATOR}_{level.lower()}_dijet_cm_ratio_fit_pt_{low:g}_{high:g}'
        canvas = ROOT.TCanvas(
            f'c_{tag}', '', DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
        )
        set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
        canvas.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
        canvas.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)

        fitted_histograms = {}
        fit_functions = {}
        fit_results = {}
        legend = ROOT.TLegend(0.45, 0.18, 0.8, 0.32)
        set_legend_style(legend)
        direction_style_indices = {'Pb-going': 1, 'p-going': 0}

        for index, orientation in enumerate(('Pb-going', 'p-going')):
            source_histogram = source_result['ratios'][orientation]
            histogram_name = (
                f'h_{GENERATOR}_{level.lower()}_dijet_cm_{orientation.lower().replace("-", "")}_'
                f'to_combined_pt_{low:g}_{high:g}'
            ).replace('.', 'p')
            histogram = source_histogram.Clone(histogram_name)
            histogram.SetDirectory(0)
            histogram.SetTitle('')
            set_1d_style(histogram, direction_style_indices[orientation])
            histogram.GetXaxis().SetTitle(eta_axis_title('dijet', 'cm'))
            histogram.GetYaxis().SetTitle('Ratio to combined')
            histogram.GetYaxis().SetRangeUser(*RATIO_RANGE)
            style_single_panel_axes(histogram)
            apply_x_range((histogram,), FRAME_CONFIG['cm']['x_range'])

            axis = histogram.GetXaxis()
            fit_low, fit_high = DIJET_CM_RATIO_FIT_RANGE or (axis.GetXmin(), axis.GetXmax())
            fit = ROOT.TF1(
                histogram.GetName() + '_fit', DIJET_CM_RATIO_FIT_FUNCTION, fit_low, fit_high,
            )
            fit.SetLineColor(histogram.GetLineColor())
            fit.SetParameters(1.0, 0.0, 0.0)  
            fit.SetLineWidth(2)
            fit_result = histogram.Fit(
                fit, DIJET_CM_RATIO_FIT_OPTIONS, '', fit_low, fit_high,
            )
            if int(fit_result) != 0:
                raise RuntimeError(
                    f'Fit failed with status {int(fit_result)} for {histogram.GetName()}'
                )
            histogram.Draw('E1' if index == 0 else 'E1 SAME')
            fit.Draw('SAME')
            legend.AddEntry(histogram, f'{orientation}/combined', 'p')
            fitted_histograms[orientation] = histogram
            fit_functions[orientation] = fit
            fit_results[orientation] = fit_result

        legend.Draw()
        selection_labels = draw_text_block(
            canvas, plot_label_lines(level, 'dijet', ptave_range, 'cm'),
            text_size=DIJET_CM_RATIO_SELECTION_TEXT_SIZE,
        )
        fit_labels = []
        fit_label = ROOT.TLatex()
        fit_label.SetNDC(True)
        fit_label.SetTextFont(DEFAULT_PLOT_STYLE.font)
        fit_label.SetTextSize(0.02)
        for index, orientation in enumerate(('Pb-going', 'p-going')):
            fit = fit_functions[orientation]
            parameters = ', '.join(
                f'p_{{{parameter}}} = {fit.GetParameter(parameter):.4f} #pm '
                f'{fit.GetParError(parameter):.4f}'
                for parameter in range(fit.GetNpar())
            )
            chi2_ndf = f'#chi^{{2}}/ndf = {fit.GetChisquare():.2f}/{fit.GetNDF()}'
            label = fit_label.DrawLatex(
                0.43, 0.87 - 0.075 * index, f'{orientation}: {parameters}, {chi2_ndf}',
            )
            label.SetTextColor(fitted_histograms[orientation].GetLineColor())
            fit_labels.append(label)

        canvas.Modified()
        canvas.Update()
        save_canvas(canvas, OUTPUT_DIR / f'{tag}.pdf', save_png=SAVE_PNG)
        canvas._dijet_cm_ratio_fit_objects = [
            legend, fit_label, *selection_labels, *fit_labels,
            *fitted_histograms.values(), *fit_functions.values(),
        ]
        result_key = (level, ptave_range)
        dijet_cm_ratio_fit_results[result_key] = {
            'canvas': canvas, 'histograms': fitted_histograms,
            'fits': fit_functions, 'fit_results': fit_results,
        }
        display(canvas)

## Unflipped, flipped, and CM frames for one beam orientation

For `FRAME_OVERLAY_DIRECTION` and `FRAME_OVERLAY_LEVEL`, overlay the three eta definitions from the same ROOT file. These curves use different coordinate definitions, so this section intentionally does not form ratios between them. The common X/Y rebin factors and normalization configured above are applied before every projection.

In [ ]:
if FRAME_OVERLAY_DIRECTION not in ('Pbgoing',  'pgoing'):
    raise ValueError('FRAME_OVERLAY_DIRECTION must be pgoing or Pbgoing')
if FRAME_OVERLAY_LEVEL not in ('Gen', 'Reco'):
    raise ValueError('FRAME_OVERLAY_LEVEL must be Gen or Reco')

FRAME_OVERLAY_CONFIG = {
    'Lab unflipped': 'PtEtaLabUnflipped',
    'Lab flipped': 'PtEtaLab',
    'CM': 'PtEtaCM',
}
_frame_overlay_objects = []

def frame_overlay_histograms(jet_kind, pt_range):
    object_name = 'InclusiveJet' if jet_kind == 'single' else 'Dijet'
    histogram_specs = [
        (
            label, input_files[FRAME_OVERLAY_DIRECTION],
            f'h{FRAME_OVERLAY_LEVEL}{object_name}{suffix}',
        )
        for label, suffix in FRAME_OVERLAY_CONFIG.items()
    ]
    return root_eta_projections(
        histogram_specs, pt_range,
        f'frame_overlay_{FRAME_OVERLAY_LEVEL}_{jet_kind}',
    )

def draw_frame_overlay(jet_kind, pt_range):
    histograms, keys = frame_overlay_histograms(jet_kind, pt_range)
    suffix = len(_frame_overlay_objects)
    canvas = ROOT.TCanvas(
        f'frame_overlay_{suffix}', '',
        DEFAULT_PLOT_STYLE.canvas_width, DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetLeftMargin(DEFAULT_PLOT_STYLE.single_panel_left_margin)
    canvas.SetBottomMargin(DEFAULT_PLOT_STYLE.single_panel_bottom_margin)
    legend = ROOT.TLegend(0.62, 0.70, 0.88, 0.88)
    set_legend_style(legend)
    maximum = max(
        histogram.GetBinContent(bin_index) + histogram.GetBinError(bin_index)
        for histogram in histograms.values()
        for bin_index in range(1, histogram.GetNbinsX() + 1)
    )
    low, high = pt_range
    frame_style_indices = {'Lab unflipped': 2, 'Lab flipped': 1, 'CM': 0}
    for index, (label, histogram) in enumerate(histograms.items()):
        set_1d_style(histogram, frame_style_indices[label])
        histogram.SetTitle('')
        histogram.GetXaxis().SetTitle(eta_axis_title(jet_kind))
        histogram.GetYaxis().SetTitle(eta_y_axis_title(jet_kind))
        style_single_panel_axes(histogram)
        histogram.SetMaximum(plot_headroom(jet_kind) * maximum)
        histogram.Draw('E1' if index == 0 else 'E1 SAME')
        legend.AddEntry(histogram, label, 'p')
    legend.Draw()
    apply_x_range(histograms.values(), FRAME_CONFIG['cm']['x_range'])
    labels = draw_text_block(
        canvas, plot_label_lines(FRAME_OVERLAY_LEVEL, jet_kind, pt_range),
    )
    tag = (f'{GENERATOR}_{FRAME_OVERLAY_DIRECTION}_{FRAME_OVERLAY_LEVEL.lower()}_'
           f'{jet_kind}_frameOverlay_pt_{low:g}_{high:g}')
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, OUTPUT_DIR / f'{tag}.pdf', save_png=SAVE_PNG)
    canvas._frame_overlay_objects = [legend, *histograms.values(), *labels]
    _frame_overlay_objects.append(canvas)
    return {'canvas': canvas, 'histograms': histograms, 'keys': keys}

In [ ]:
frame_overlay_results = {}
for jet_kind, pt_ranges in (
    ('single', SINGLE_JET_PT_BINS),
    ('dijet', DIJET_PTAVE_BINS),
):
    for pt_range in pt_ranges:
        result_key = (FRAME_OVERLAY_DIRECTION, FRAME_OVERLAY_LEVEL, jet_kind, pt_range)
        result = draw_frame_overlay(jet_kind, pt_range)
        frame_overlay_results[result_key] = result
        print(result_key, result['keys'])
        display(result['canvas'])